# Simple Model Comparison

This notebook gives a quick first comparison of four model types. Each model predicts the two product purities and two energy duties from the same DWSIM data.

This is an exploratory comparison. The final result comes from `main.py`, which uses cross-validation, a separate final check set, and material-balance correction.

In [ ]:
from pathlib import Path
from time import perf_counter
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Load the data

The model uses the feed stage as a fraction of the column height. The integer feed stage is kept in the dataset for reference, but using both would give the model the same information twice.

In [ ]:
FEATURES = [
    "feed_temperature_K",
    "feed_pressure_Pa",
    "benzene_feed_fraction",
    "number_of_stages",
    "feed_stage_fraction",
    "reflux_ratio",
    "bottoms_flow_mol_s",
]
TARGETS = [
    "distillate_benzene_purity",
    "bottoms_toluene_purity",
    "condenser_duty",
    "reboiler_duty",
]

dataset_path = Path("dataset.csv")
if not dataset_path.exists():
    dataset_path = Path("../dataset.csv")

data = pd.read_csv(dataset_path)
required = FEATURES + TARGETS
missing = sorted(set(required) - set(data.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")
if data[required].isna().any().any() or not np.isfinite(data[required]).all().all():
    raise ValueError("The model columns contain missing or non-finite values")
if data.duplicated(subset=FEATURES).any():
    raise ValueError("The dataset contains duplicate model inputs")

X_train, X_test, y_train, y_test = train_test_split(
    data[FEATURES], data[TARGETS], test_size=0.20, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## Compare four model types

The comparison covers a polynomial model, a tree model, gradient boosting, and a neural network. Scaling is done inside each model, so it learns only from the training rows.

In [ ]:
def scaled_model(model):
    return TransformedTargetRegressor(regressor=model, transformer=StandardScaler())


models = {
    "Polynomial Ridge": scaled_model(
        Pipeline([
            ("scale", StandardScaler()),
            ("polynomial", PolynomialFeatures(2, include_bias=False)),
            ("ridge", Ridge(alpha=1e-3)),
        ])
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "CatBoost": MultiOutputRegressor(
        CatBoostRegressor(
            iterations=500, depth=7, learning_rate=0.05,
            loss_function="RMSE", random_seed=RANDOM_STATE,
            verbose=False, allow_writing_files=False,
        )
    ),
    "ANN 128-64": scaled_model(
        Pipeline([
            ("scale", StandardScaler()),
            ("ann", MLPRegressor(
                hidden_layer_sizes=(128, 64), early_stopping=True,
                validation_fraction=0.15, n_iter_no_change=30,
                max_iter=600, random_state=RANDOM_STATE,
            )),
        ])
    ),
}

In [ ]:
records = []
predictions = {}
target_ranges = np.ptp(y_train.to_numpy(), axis=0)

for model_name, model in models.items():
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - start
    predicted = np.asarray(model.predict(X_test))
    predictions[model_name] = predicted

    for index, target in enumerate(TARGETS):
        actual = y_test.iloc[:, index]
        rmse = np.sqrt(mean_squared_error(actual, predicted[:, index]))
        records.append({
            "model": model_name,
            "target": target,
            "MAE": mean_absolute_error(actual, predicted[:, index]),
            "RMSE": rmse,
            "R2": r2_score(actual, predicted[:, index]),
            "NRMSE": rmse / target_ranges[index],
            "fit_seconds": fit_seconds,
        })

metrics = pd.DataFrame(records)
ranking = (
    metrics.groupby("model", as_index=False)
    .agg(mean_NRMSE=("NRMSE", "mean"), mean_R2=("R2", "mean"), fit_seconds=("fit_seconds", "first"))
    .sort_values("mean_NRMSE")
    .reset_index(drop=True)
)
ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
display(ranking.round(6))
display(metrics.sort_values(["target", "NRMSE"]).round(6))

## Basic physical check

The first two outputs are fractions, so they must stay between 0 and 1. The final pipeline also enforces the benzene material balance; this notebook only reports raw baseline behavior.

In [ ]:
checks = []
for model_name, predicted in predictions.items():
    invalid = (predicted[:, :2] < 0) | (predicted[:, :2] > 1)
    checks.append({
        "model": model_name,
        "invalid_distillate_predictions": int(invalid[:, 0].sum()),
        "invalid_bottoms_predictions": int(invalid[:, 1].sum()),
    })
display(pd.DataFrame(checks))

In [ ]:
best_name = ranking.iloc[0]["model"]
best_predictions = predictions[best_name]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for index, ax in enumerate(axes.flat):
    actual = y_test.iloc[:, index].to_numpy()
    predicted = best_predictions[:, index]
    low = min(actual.min(), predicted.min())
    high = max(actual.max(), predicted.max())
    ax.scatter(actual, predicted, s=10, alpha=0.4)
    ax.plot([low, high], [low, high], "--", color="black")
    ax.set_title(TARGETS[index].replace("_", " "))
    ax.set_xlabel("DWSIM")
    ax.set_ylabel("Model")

fig.suptitle(f"Best baseline: {best_name}")
fig.tight_layout()
plt.show()

## What this notebook shows

A low error here means the model can reproduce held-back DWSIM cases from the same sampled range. It does not prove plant accuracy or safe use outside that range. See `main.py` and `Results_Summary.pdf` for the final model selection and stronger checks.